In [23]:
import torch
import torch.nn as nn
import math

In [24]:
patch_embed = nn.Conv2d(
    in_channels=3,
    out_channels=96,
    kernel_size=4,
    stride=4
)

x = torch.randn(2, 3, 224, 224)

y = patch_embed(x)

print(y.shape)

torch.Size([2, 96, 56, 56])


In [25]:
y = y.flatten(2)
y = y.transpose(1,2)

print(y.shape)

torch.Size([2, 3136, 96])


In [26]:
class patch_embed(nn.Module):
    def __init__(self, in_channels, emb_dim, kernel_size= 4, stride= 4):
        super().__init__()

        self.proj = nn.Conv2d(in_channels= in_channels, out_channels= emb_dim, kernel_size= kernel_size, stride= stride)
        self.norm = nn.LayerNorm(emb_dim)
    def forward(self, X):
        X = self.proj(X)
        X = X.flatten(2)
        X = X.transpose(1,2)
        X = self.norm(X)
        
        return X

In [27]:
def window_partition(X, window_size):
    
    B,H,W,C = X.shape

    X = X.reshape(
         B,
        H // window_size,
        window_size,
        W // window_size,
        window_size,
        C
    )

    X = X.permute(0,1,3,2,4,5).contiguous()

    window = X.reshape(-1, window_size, window_size, C)

    return window


In [28]:
def window_reverse(window, window_size, H, W, B):
    C = window.shape[-1]

    x = window.reshape(
        B,
        H // window_size,
        W // window_size,
        window_size,
        window_size,
        C
    )

    x = x.permute(0,1,3,2,4,5).contiguous()

    x = x.reshape(B, H, W, C)

    return x

In [29]:
# window_partition ile window_reverse doğru çalışıyor mu diye test
B, H, W, C = 2, 56, 56, 96
window_size = 7

x = torch.randn(B, H, W, C)

windows = window_partition(x, window_size)
print(windows.shape)
# (128, 7, 7, 96)

reconstructed = window_reverse(
    windows, window_size, H, W, B
)

print(reconstructed.shape)
# (2, 56, 56, 96)

print(torch.allclose(x, reconstructed))
# True

torch.Size([128, 7, 7, 96])
torch.Size([2, 56, 56, 96])
True


In [ ]:
class WindowAttention(nn.Module):
    def __init__(self, dim, num_heads, window_size=7):
        super().__init__()

        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.window_size = window_size

        # Q, K, V projeksiyonu
        self.qkv = nn.Linear(dim, dim * 3)

        # Attention çıktısı projeksiyonu
        self.proj = nn.Linear(dim, dim)

        # Relative Position Bias tablosu
        num_relative_positions = (2 * window_size - 1) ** 2

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros(num_relative_positions, num_heads)
        )

        # Token koordinatlarını oluştur
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)

        coords = torch.stack(
            torch.meshgrid(
                coords_h,
                coords_w,
                indexing="ij"
            )
        )
        # (2, window_size, window_size)

        coords_flatten = torch.flatten(coords, 1)
        # (2, N)

        # Her token çifti arasındaki göreli koordinatlar
        relative_coords = (
            coords_flatten[:, :, None]
            - coords_flatten[:, None, :]
        )
        # (2, N, N)

        relative_coords = relative_coords.permute(
            1, 2, 0
        ).contiguous()
        # (N, N, 2)

        # Negatif indeksleri pozitif aralığa taşı
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1

        # İki boyutlu koordinatı tek indekse dönüştür
        relative_coords[:, :, 0] *= 2 * window_size - 1

        relative_position_index = relative_coords.sum(dim=-1)
        # (N, N)

        # Model parametresi değil, sabit indeks bilgisi
        self.register_buffer(
            "relative_position_index",
            relative_position_index
        )

        nn.init.trunc_normal_(
            self.relative_position_bias_table,
            std=0.02
        )

    def forward(self, X, mask= None):
        # X: (B_windows, N, C)
        B_windows, N, C = X.shape

        # Q, K, V üret
        qkv = self.qkv(X)
        # (B_windows, N, 3*C)

        qkv = qkv.reshape(
            B_windows,
            N,
            3,
            self.num_heads,
            self.head_dim
        )
        # (B_windows, N, 3, heads, head_dim)

        qkv = qkv.permute(2, 0, 3, 1, 4)
        # (3, B_windows, heads, N, head_dim)

        q, k, v = qkv[0], qkv[1], qkv[2]
        # Her biri: (B_windows, heads, N, head_dim)

        # Attention skorları
        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        # (B_windows, heads, N, N)

        # Relative Position Bias tablosundan değerleri al
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.reshape(-1)
        ]
        # (N*N, heads)

        relative_position_bias = relative_position_bias.reshape(
            N, N, self.num_heads
        )
        # (N, N, heads)

        relative_position_bias = relative_position_bias.permute(
            2, 0, 1
        ).contiguous()
        # (heads, N, N)

        # Bias'ı attention skorlarına ekle
        scores = scores + relative_position_bias.unsqueeze(0)

        if mask is not None:
            # mask: (num_windows, N, N)
            num_windows = mask.shape[0]

            scores = scores.reshape(
                B_windows // num_windows,
                num_windows,
                self.num_heads,
                N,
                N
            )

            scores = scores + mask.unsqueeze(0).unsqueeze(2)

            scores = scores.reshape(
                B_windows,
                self.num_heads,
                N,
                N
            )

        attn = scores.softmax(dim=-1)

        # Attention ağırlıklarıyla V'yi birleştir
        out = attn @ v
        # (B_windows, heads, N, head_dim)

        # Head'leri birleştir
        out = out.transpose(1, 2).reshape(
            B_windows, N, C
        )
        # (B_windows, N, C)

        out = self.proj(out)

        return out

In [34]:
def create_shifted_window_mask(
    H,
    W,
    window_size=7,
    shift_size=3
):
    img_mask = torch.zeros((1, H, W, 1))

    h_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None)
    )

    w_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None)
    )

    count = 0

    for h in h_slices:
        for w in w_slices:
            img_mask[:, h, w, :] = count
            count += 1

    mask_windows = window_partition(
        img_mask, window_size
    )

    mask_windows = mask_windows.reshape(
        -1, window_size * window_size
    )

    attn_mask = (
        mask_windows.unsqueeze(1)
        - mask_windows.unsqueeze(2)
    )

    attn_mask = attn_mask.masked_fill(
        attn_mask != 0, -100.0
    )

    attn_mask = attn_mask.masked_fill(
        attn_mask == 0, 0.0
    )

    return attn_mask

In [35]:
def shifted_window_attention(
    x,
    attention,
    window_size=7,
    shift_size=3
):
    B, H, W, C = x.shape

    # 1. Feature map'i kaydır
    shifted_x = torch.roll(
        x,
        shifts=(-shift_size, -shift_size),
        dims=(1, 2)
    )

    # 2. Kaydırılmış feature map için mask oluştur
    mask = create_shifted_window_mask(
        H, W,
        window_size=window_size,
        shift_size=shift_size
    ).to(device=x.device, dtype=x.dtype)

    # 3. Pencerelere ayır
    windows = window_partition(
        shifted_x, window_size
    )

    windows = windows.reshape(
        -1, window_size * window_size, C
    )

    # 4. Maskeli attention
    attended_windows = attention(
        windows, mask=mask
    )

    # 5. Pencereleri birleştir
    attended_windows = attended_windows.reshape(
        -1, window_size, window_size, C
    )

    shifted_x = window_reverse(
        attended_windows,
        window_size,
        H, W, B
    )

    # 6. Kaydırmayı geri al
    x = torch.roll(
        shifted_x,
        shifts=(shift_size, shift_size),
        dims=(1, 2)
    )

    return x

In [36]:
class SwinTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        window_size=7,
        shift_size=0,
        mlp_ratio=4.0
    ):
        super().__init__()

        self.dim = dim
        self.window_size = window_size
        self.shift_size = shift_size

        # 1. Layer Normalization
        self.norm1 = nn.LayerNorm(dim)

        # 2. Window-based Multi-Head Self-Attention
        self.attn = WindowAttention(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size
        )

        # 3. Second Layer Normalization
        self.norm2 = nn.LayerNorm(dim)

        # 4. MLP
        hidden_dim = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        # x: (B, H, W, C)

        B, H, W, C = x.shape

        # Attention residual connection
        shortcut = x
        x = self.norm1(x)

        if self.shift_size > 0:
            # Shifted Window Attention
            attn_out = shifted_window_attention(
                x,
                attention=self.attn,
                window_size=self.window_size,
                shift_size=self.shift_size
            )
        else:
            # Normal Window Attention
            windows = window_partition(
                x,
                self.window_size
            )

            windows = windows.reshape(
                -1,
                self.window_size * self.window_size,
                C
            )

            attn_windows = self.attn(windows)

            attn_windows = attn_windows.reshape(
                -1,
                self.window_size,
                self.window_size,
                C
            )

            attn_out = window_reverse(
                attn_windows,
                self.window_size,
                H,
                W,
                B
            )

        x = shortcut + attn_out

        # MLP residual connection
        x = x + self.mlp(self.norm2(x))

        return x


In [37]:
class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()

        # 4 komşu token birleştirildiği için 4C özellik oluşur.
        self.norm = nn.LayerNorm(4 * dim)

        # Kanal sayısını 4C'den 2C'ye indir.
        self.reduction = nn.Linear(
            4 * dim,
            2 * dim,
            bias=False
        )

    def forward(self, x):
        # x: (B, H, W, C)
        B, H, W, C = x.shape

        assert H % 2 == 0 and W % 2 == 0, \
            "H ve W çift olmalı!"

        # Dört farklı konumdaki komşu token'ları al.
        x0 = x[:, 0::2, 0::2, :]  # (B, H/2, W/2, C)
        x1 = x[:, 1::2, 0::2, :]  # (B, H/2, W/2, C)
        x2 = x[:, 0::2, 1::2, :]  # (B, H/2, W/2, C)
        x3 = x[:, 1::2, 1::2, :]  # (B, H/2, W/2, C)

        # Kanal boyutunda birleştir.
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        # (B, H/2, W/2, 4C)

        x = self.norm(x)
        x = self.reduction(x)
        # (B, H/2, W/2, 2C)

        return x


In [38]:
class PatchEmbed(nn.Module):
    def __init__(self, in_channels=3, embed_dim=96):
        super().__init__()

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=4,
            stride=4
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # (B, 3, 224, 224)
        x = self.proj(x)
        # (B, 96, 56, 56)

        x = x.permute(0, 2, 3, 1)
        # (B, 56, 56, 96)

        x = self.norm(x)

        return x

In [39]:
class SwinStage(nn.Module):
    def __init__(
        self,
        dim,
        depth,
        num_heads,
        window_size=7,
        downsample=True
    ):
        super().__init__()

        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                window_size=window_size,
                shift_size=0 if i % 2 == 0 else window_size // 2
            )
            for i in range(depth)
        ])

        self.downsample = (
            PatchMerging(dim) if downsample else nn.Identity()
        )

    def forward(self, x):
        for block in self.blocks:
            x = block(x)

        x = self.downsample(x)

        return x

In [40]:
class SwinTransformer(nn.Module):
    def __init__(
        self,
        num_classes=1000,
        embed_dim=96,
        depths=(2, 2, 6, 2),
        num_heads=(3, 6, 12, 24),
        window_size=7
    ):
        super().__init__()

        self.patch_embed = PatchEmbed(
            in_channels=3,
            embed_dim=embed_dim
        )

        dims = [
            embed_dim,
            embed_dim * 2,
            embed_dim * 4,
            embed_dim * 8
        ]

        self.stages = nn.ModuleList()

        for i in range(4):
            self.stages.append(
                SwinStage(
                    dim=dims[i],
                    depth=depths[i],
                    num_heads=num_heads[i],
                    window_size=window_size,
                    downsample=(i < 3)
                )
            )

        self.norm = nn.LayerNorm(dims[-1])
        self.head = nn.Linear(dims[-1], num_classes)

    def forward(self, x):
        # (B, 3, 224, 224)
        x = self.patch_embed(x)
        # (B, 56, 56, 96)

        for stage in self.stages:
            x = stage(x)

        # (B, 7, 7, 768)
        x = self.norm(x)

        # Spatial global average pooling
        x = x.mean(dim=(1, 2))
        # (B, 768)

        x = self.head(x)
        # (B, num_classes)

        return x

In [41]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SwinTransformer(
    num_classes=10,
    embed_dim=48,
    depths=(2, 2, 2, 2),
    num_heads=(3, 6, 12, 24)
).to(device)

x = torch.randn(1, 3, 224, 224).to(device)

model.eval()

with torch.no_grad():
    output = model(x)

print("Output shape:", output.shape)

TypeError: WindowAttention.forward() got an unexpected keyword argument 'mask'